In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

from config import DATA_DIR, CHROMA_DIR, CHUNK_SIZE, CHUNK_OVERLAP
from rag.loader   import load_pdfs
from rag.chunker  import chunk_documents
from rag.store    import get_chroma_collection, add_documents_to_db, clear_collection
from rag.embedder import get_embedding_function

print('All modules loaded ✅')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'CHROMA_DIR : {CHROMA_DIR}')

In [ ]:
# ── Configure these before running ────────────────────────────────────────
PDF_SOURCE_DIR   = DATA_DIR   # ← Path to your PDFs
CLEAR_EXISTING   = False      # ← Set True to wipe DB before ingesting
BATCH_SIZE       = 64         # ← DB insert batch size
# ──────────────────────────────────────────────────────────────────────────

print(f'PDF source  : {PDF_SOURCE_DIR}')
print(f'Clear DB    : {CLEAR_EXISTING}')
print(f'Chunk size  : {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars')

In [ ]:
t0 = time.time()
documents = load_pdfs(PDF_SOURCE_DIR)
t1 = time.time()

print(f'\n⏱ Loaded in {t1-t0:.2f}s')
print(f'📝 Pages extracted : {len(documents)}')

# Summary table
from collections import Counter
page_counts = Counter(d['source'] for d in documents)
print(f'\n{"File":<40} {"Pages"}')
print('-' * 50)
for src, cnt in page_counts.items():
    print(f'{src:<40} {cnt}')

In [ ]:
if not documents:
    raise RuntimeError('No documents loaded. Place PDFs in the /data/ directory first.')

t0 = time.time()
chunks = chunk_documents(documents)
t1 = time.time()

print(f'⏱ Chunked in {t1-t0:.2f}s')
print(f'🔢 Total chunks    : {len(chunks)}')
print(f'📏 Avg chunk length: {sum(len(c["text"]) for c in chunks) / len(chunks):.0f} chars')

# Size histogram
import matplotlib.pyplot as plt
lengths = [len(c['text']) for c in chunks]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lengths, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(CHUNK_SIZE, color='red', linestyle='--', label=f'Target size ({CHUNK_SIZE})')
ax.set_xlabel('Chunk Length (chars)')
ax.set_ylabel('Count')
ax.set_title('Chunk Length Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
embed_fn = get_embedding_function()

# Embed just the first 3 chunks as a sanity check
sample_texts = [c['text'] for c in chunks[:3]]
sample_vecs  = embed_fn.embed_texts(sample_texts)

print(f'✅ Sample embedding shape : ({len(sample_vecs)}, {len(sample_vecs[0])})')
print(f'   Min value : {min(v for vec in sample_vecs for v in vec):.4f}')
print(f'   Max value : {max(v for vec in sample_vecs for v in vec):.4f}')

In [ ]:
collection = get_chroma_collection(CHROMA_DIR)
print(f'Docs before ingestion : {collection.count()}')

if CLEAR_EXISTING:
    print('🗑 Clearing existing data…')
    clear_collection(collection)

t0 = time.time()
add_documents_to_db(chunks, collection, batch_size=BATCH_SIZE)
t1 = time.time()

print(f'\n⏱ Stored in {t1-t0:.2f}s')
print(f'✅ Docs after ingestion : {collection.count()}')

In [ ]:
from rag.retriever import retrieve_context

TEST_QUERIES = [
    'What programs does Pradita University offer?',
    'How do I apply for admission?',
    'What is the tuition fee structure?',
]

for q in TEST_QUERIES:
    result = retrieve_context(q, top_k=3, collection=collection)
    print(f'\n🔍 Query: {q}')
    print(f'   Sources: {result["sources"]}')
    print(f'   Chunks:  {len(result["chunks"])}')
    if result['chunks']:
        best = result['chunks'][0]
        print(f'   Best match (dist={best["distance"]:.4f}): {best["text"][:100]}...')